# Description
train.csv - the training set (contains 1458644 trip records)
test.csv - the testing set (contains 625134 trip records)

# Attributes
- ### id - a unique identifier for each trip (not needed)
- ### vendor_id - a code indicating the provider associated with the trip record (not needed)
- ### pickup_datetime - date and time when the meter was engaged
- ### dropoff_datetime - date and time when the meter was disengaged
- ### passenger_count - the number of passengers in the vehicle (driver entered value)
- ### pickup_longitude - the longitude where the meter was engaged
- ### pickup_latitude - the latitude where the meter was engaged
- ### dropoff_longitude - the longitude where the meter was disengaged
- ### dropoff_latitude - the latitude where the meter was disengaged
- ### store_and_fwd_flag - This flag indicates whether the trip record was held in vehicle memory before sending to the vendor because the vehicle did not have a connection to the server - Y=store and forward; N=not a store and forward trip (not needed)
- ### trip_duration - duration of the trip in seconds

## AIM OF THE PROJECT 
our aim is to make a prediction of the duration of the taxi trip, we are in New York, it's a metropoly and there are specific time where the roads gets specially busy, while in other time empty and the taxi will get less time to arrive to destination. So something that will very important for us will be pickup_datetime and dropoff_datetime as our main reference for the duration of the trip. 
- Then also the neighbour will be a matter of how crowded may get more time than other to arrive at destination, so for example it a taxi start from a specific location we know that will take more time because it's in that specific area at that specific time. 

In [129]:
#importing the libraries
import osmnx as ox
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 
import folium 
from folium.plugins import HeatMap
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import networkx as nx
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from sklearn.metrics import mean_squared_error
import plotnine as p9

In [ ]:
#downloading the dataset 
train = pd.read_csv('/Users/carlofiammenghi/train.csv')

#to work on it later
prediction = pd.read_csv('/Users/carlofiammenghi/test.csv')

train['is_test'] = 0 
prediction['is_test'] = 1

prediction['trip_duration'] = -1

df = pd.concat([train, prediction], ignore_index=True)

# let's convert the format in date
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# let's create the new numerical columns (that the model accept)
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.weekday
df['month'] = df['pickup_datetime'].dt.month
    
#----------------------------------------------------------------------------------------
# NEW COLUMNS WHICH WILL GIVE MORE DEPTH AND UNDERSTANDING TO OUR MODEL !!!
# let's also create the haversine distance, foundamental for XGboost
def haversine_distance(lat1, lon1, lat2, lon2):
    r= 6371.0 #earth's ray in km x angle 

    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat /2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return r * c


#----------------------------------------------------------------------------------------
# manhattan distance in Km 
def manhattan_distance(lat1, lon1, lat2, lon2):
    dist_lat = np.abs(lat2 - lat1)* 111.0 # New York (~40.7 lati.) 1 degree are ~111km
    dist_lon =np.abs(lon2 -lon1)* 84.5    # 1 degree are ~84.5 Km
   
    return dist_lat + dist_lon


#----------------------------------------------------------------------------------------
# it's the compass for which direction the taxi is starting and going to 
def bearing_calculator(lat1,lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1

    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    
    initial_bearing = np.arctan2(x,y)
    
    return np.degrees(initial_bearing)

#----------------------------------------------------------------------------------------


df['distance_km'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

print('calculating new features on train...')
df['manhattan_dist'] = manhattan_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

df['bearing'] = bearing_calculator(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'Y':1, 'N':0})

print(f'new features calculated on {len(df)} rows')


In [131]:
df['speed_km_h'] = np.where(
    df['is_test'] == 0,
    df['manhattan_dist'] / (df['trip_duration'] / 3600),
    np.nan # the test set doensn't have a real duration
)

### the golden feature: <br> <b>(prediction_trip_duration, simply distance / avg_speed_for_hour)</b>: <br> this way we can mesaure the expected trip duration also in the test set give the location and at what time the taxi started the ride 🤩

In [ ]:
good_data = df[(df['is_test']==0) & (df['speed_km_h'] > 0) & (df['speed_km_h'] <=130)]

avg_speed_per_hour = good_data.groupby('hour')['speed_km_h'].mean()

overall_avg_speed = good_data['speed_km_h'].mean()

print(f"Velocità media di NY {overall_avg_speed:.2f} km/h")

df['avg_speed_per_hour'] = df['hour'].map(avg_speed_per_hour)

df['estimated_trip_duration'] = (df['manhattan_dist'] / df['avg_speed_per_hour']) * 3600

df = df[((df['speed_km_h'] > 0) & (df['speed_km_h'] <=130))| (df['is_test'] ==1)]

df.head()


In [133]:
# which has been substituded by hour, day of week and distance_in_km 
col_to_drop = ['pickup_datetime']
df = df.drop(columns=col_to_drop)

In [ ]:
print(df.info())

In [ ]:
train_only = df[df['is_test'] == 0].sample(n=100000, random_state=42)

for col in df.select_dtypes(include='number').columns:
    if col in ['is_test', 'avg_speed_per_hour']:
        continue
    plt.figure()
    
    if pd.api.types.is_integer_dtype(df[col]) and df [col].nunique() <= 50:
            df[col].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
    elif 'speed' in col:
        plt.hist(df[col].between(0,130).pipe(lambda m: df.loc[m,col]), bins=45, color='coral', edgecolor='black')
        plt.xlim(0,130)
    else:
        plt.hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='black')
    plt.yscale('log'); plt.title(f'{col} distribution'); plt.xlabel(col); plt.ylabel('count (log)')
    plt.tight_layout(); plt.show()

plt.figure(figsize=(10,4))
avg_speed_per_hour.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Average speed per hour')
plt.xlabel('Hour')
plt.ylabel('speed (km/h)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.style.use('ggplot')
map_vendorid = (
    p9.ggplot(df, p9.aes(x='trip_duration', fill='factor(vendor_id)'))
    + p9.geom_density(position='stack')
    + p9.scale_x_log10()
)

map_vendorid

In [ ]:
hist_vendor =(
    p9.ggplot(df, p9.aes(x='factor(passenger_count)', fill='factor(vendor_id)'))
    + p9.geom_bar(position='stack',alpha=0.8)
    + p9.theme_minimal()
    + p9.scale_fill_manual(values={1:"orange", 2:"blue"})

   # +p9.scale_minimal()
    +p9.labs(title='diffenrece between passenger count forvendor 1/2',
        x='vendor ID',
        y="number of passengers",
        fill='company (vendor_id)')
)
hist_vendor

In [ ]:
m = folium.Map(location=[40.7306, -73.9866], zoom_start=11)

data_pickup = df[['pickup_longitude', 'pickup_latitude']].values.tolist()
data_dropoff = df[['dropoff_longitude', 'dropoff_latitude']].values.tolist()    

fg_pickup = folium.FeatureGroup(name="Heatmap Pickup")
fg_dropoff = folium.FeatureGroup(name="Heatmap Dropoff")

HeatMap(data_pickup, radius=8, blur=12, gradient={0.4:"blue", 0.65:"lime", 1:"cyan"}).add_to(fg_pickup) 
HeatMap(data_dropoff, radius=8, blur=12, gradient={0.4:"orange", 0.65:"red", 1:"darkred"}).add_to(fg_dropoff) 

fg_pickup.add_to(m)
fg_dropoff.add_to(m)

folium.LayerControl().add_to(m)

display(m)

now that we added many features to help later the model to predict with more accuracy the trip duration, now we split the data and go towards the model training 


In [ ]:
train_set = df[df['is_test']==0 ].copy()
test_set = df[df['is_test']==1 ].copy()

train_set = train_set.drop(columns=['is_test'])
test_set = test_set.drop(columns=['is_test'])

if 'trip_duration' in test_set.columns:
    test_set = test_set.drop(columns=['trip_duration'])

# which has been substituded by hour, day of week and distance_in_km 
col_to_drop = ['id']
train_set = train_set.drop(columns=col_to_drop)

#-------------------------------------------------------------------------------------------------------------
train_set, internal_test = train_test_split(train_set, test_size= 0.20, random_state=42)

print(f'training: {len(train_set)} rows')
print(f"internal test: {len(internal_test)} rows")
print(f'test: {len(test_set)} rows')

# let's clean duration time 

In [ ]:
plt.figure(figsize=(12,6))

sns.histplot(np.log1p(df['trip_duration']),kde=True, color='orange')

plt.title('Log distribution of trip duration')
plt.xlabel('Log(last in seconds)')
plt.show()

In [ ]:
lower_limit = np.exp(4) # 33 secs +-
upper_limit = np.exp(9)  # 8.103 secs (~5.5 hrs)

too_short = train_set[train_set['trip_duration'] < lower_limit]
too_long = train_set[train_set['trip_duration'] > upper_limit]

n_short = len(too_short)
n_long = len(too_long)

print(f"total rows now:{len(train_set)}")
print(f"ride too short (< 33s){n_short} ({ (n_short/len(train_set))*100:.4f}%)")
print(f"ride too long (>6h): {n_long} ({ (n_long/len(train_set))*100:.4f}%)")


In [ ]:
train_temp_new = train_set[
    (train_set["trip_duration"] >= lower_limit) &
    (train_set["trip_duration"] <= upper_limit)
]


plt.figure(figsize=(12,6))

sns.histplot(np.log1p(train_temp_new['trip_duration']),kde=True, color='orange')

plt.title(' Log distribution of trip duration')
plt.xlabel('Log(last in seconds)')
plt.show()

# 1. let's remove the longitude and latidue outliers

In [ ]:
# 1. Definition of borders (widely)
lat_min, lat_max = 40.15, 41.30
lon_min, lon_max = -75.50, -73.30

train_temp_new_1 = train_temp_new[
    (train_temp_new['pickup_latitude'].between(lat_min, lat_max)) & 
    (train_temp_new["pickup_longitude"].between(lon_min, lon_max)) &
    (train_temp_new['dropoff_latitude'].between(lat_min, lat_max)) & 
    (train_temp_new["dropoff_longitude"].between(lon_min, lon_max))
]

print(f"Total rows:{len(train_set)}")
print(f"Outliers nei Pickup: {(len(train_set) - len(train_temp_new_1))}")
print(f"percentuale outliers: {(len(train_set) - len(train_temp_new_1)) / len(train_set) * 100:2f}%")

# train_set = train_clean

In [ ]:
# create map centred on m
m = folium.Map(location=[40.7128, -74.0060], zoom_start=7)

# preparing data
data_pickup = train_temp_new_1[['pickup_latitude', 'pickup_longitude']].values.tolist()
data_dropoff = train_temp_new_1[['dropoff_latitude', 'dropoff_longitude']].values.tolist()

# create 2 groups pickup and drop off
fg_pickup = folium.FeatureGroup(name= 'Heatmap Pick-up')
fg_dropoff = folium.FeatureGroup(name='Heatmap Drop-off')

# add heat map to respective groups
HeatMap(data_pickup,  radius=8, blur=12, gradient={0.4: 'blue', 0.65:'lime', 1:'cyan'}).add_to(fg_pickup)
HeatMap(data_dropoff, radius=8, blur=12, gradient={0.4: 'orange', 0.65: "red", 1:"darkred"}).add_to(fg_dropoff)

fg_pickup.add_to(m)
fg_dropoff.add_to(m)
folium.LayerControl().add_to(m)

display(m)

 split train / validation and test

In [ ]:
def null_summary(dataset):
    null_summary = dataset.isnull().sum()

    print("counting values Nan for every column")
    print(null_summary)

    if null_summary.sum() == 0:
        print("\nPerfect dataset: no NULL values")
    else:
        print("\nWarning there are NULL values")

null_summary(train_set)
null_summary(internal_test)


train, validation = train_test_split(train_set, test_size = 0.20, random_state=42)

print(f'training: {len(train)} rows')
print(f"validation: {len(validation)} rows")
print(f"test: {len(internal_test)} rows")

In [155]:
#impenmenting callback like linear decay in NNs

class LinearDecayLR(xgb.callback.TrainingCallback):
    def __init__(self, lr_start=0.05, lr_end=0.0005, n_rounds=10000, print_every=1000):
        self.lr_start=lr_start
        self.lr_end = lr_end
        self.n_rounds = n_rounds
        self.print_every = print_every
      
    def after_iteration(self, model, epoch, evals_log):
        lr = self.lr_start - (self.lr_start - self.lr_end) * (epoch / self.n_rounds)
        lr = max(lr, self.lr_end)
        model.set_param('learning_rate', lr)

        if epoch % self.print_every == 0:
            print(f' [Round {epoch}] LR:{lr:.6f}')

        return False

# let's create the XGBOOST model

In [ ]:
# --- 1 SEPARETING THE FEATURES (X) FROM TARGET (Y)
#we say what's the column with the answer 
target = 'trip_duration'

#
y_train_clipped = train[target].clip(lower=55, upper=7200)
y_val_clipped = validation[target].clip(lower=55, upper=7200)

col_to_drop = [target, 'dropoff_datetime', 'speed_km_h']

#train set
X_train = train.drop(columns=col_to_drop)
y_train = np.log1p(y_train_clipped)

#validation set
X_val = validation.drop(columns=col_to_drop)
y_val = np.log1p(y_val_clipped)

# internal Test
X_internal_test = internal_test.drop(columns=col_to_drop)
y_internal_test = internal_test[target].clip(lower=55, upper=7200)

# --- 2. MODEL CONFIGURATION 
# let's use XGBOOSTregressor becaouse we need tu guess the number in seconds, not a category
xgb_model = xgb.XGBRegressor(
    n_estimators = 5000,           # max number of 'trees' 
    learning_rate=0.02,            # speed of learning
    max_depth=9,                   # deepth of trees (complexity)
    early_stopping_rounds=450,     # if for 50 rounds don't imporove accuracy, stop rounds
    random_state=42,               # setting the seed
    subsample = 0.7,               # hide 20% of the runs from the dataset
    colsample_bytree = 0.6,        # hide 30% of columns (features)
    objective='reg:squarederror',  # squared error loss function instead of L1 which helps with outliers 
    eval_metric= 'rmse',
    reg_lambda = 1,
    reg_alpha = 7,
    #min_child_weight = 5,
    #device="mps",
    n_jobs=-1,                # use all the cpu cores
    callbacks=[LinearDecayLR(lr_start=0.02, lr_end=0.005, n_rounds=5000, print_every=500)]

)

# --- 3. TRAINING
print("starting training XGBoost...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],  # evalute the error on training and validation
    verbose=500 # print an update every 100 trees to don't clog the screen
)

# let's extract the results to print them later
results = xgb_model.evals_result()

# --- 4. MOMENT OF TRUTH (TEST SET)
# let's see the model prediction on internal test set 
forecast_log = xgb_model.predict(X_internal_test)

forecast_sec = np.expm1(forecast_log)

forecast_sec = np.clip(forecast_sec, a_min=1, a_max=7200)

real_log =np.log1p(y_internal_test)

rmsle = np.sqrt(mean_squared_error(real_log, forecast_log))

mae = mean_absolute_error(y_internal_test, forecast_sec)

print("-"*80)
print(f"RMSLE on Test Set: {rmsle:.5f}")
print(f"Last Value on Training: {results['validation_0']['rmse'][-1]:.5f}")
print(f"FINAL RESULT ON EXAM (INTERNAL TEST SET):")
print(f"Mean Absolute Error (MAE) is: {mae:.2f} seconds")

In [ ]:

#let's extract the history of errors regrested from the model
results = xgb_model.evals_result()

epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

plt.figure(figsize=(12, 6))

# training curve Blue
plt.plot(x_axis, results['validation_0']['rmse'], label='Train Error (loss)', color='blue', linewidth=3)

# validation curve Orange
plt.plot(x_axis, results['validation_1']['rmse'], label='Validation Error', color='orange', linewidth=2)

plt.title('XGBoost Learning Curve')
plt.xlabel('Num. of trees (boosting Rounds)')
plt.ylabel('Logartihm Error (RMSE')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.axvline(x=epochs-1, color='red', linestyle='--', label='End training')
plt.show()

In [ ]:
sec_forecast = np.expm1(forecast)

analysis = X_test.copy()
analysis['last_in_sec'] = y_test.values
analysis['forecast_sec'] = sec_forecast
analysis['absolute_error'] = np.abs(analysis['last_in_sec'] - analysis['forecast_sec'])

worst_errors = analysis.sort_values('absolute_error', ascending=False)

print("10 worst errors")
print('-'*80)
print(worst_errors[['distance_km', 'last_in_sec', 'forecast_sec', 'absolute_error']].head())


In [ ]:
#Transforming the prevision in real seconds, then compute the error
actual_sec = y_test
pred_sec = np.expm1(forecast_safe)
errors = actual_sec - pred_sec  #If positive model has underestimated otherwise overestimated 

plt.figure(figsize=(16, 7))

# GRAPH 1: Actual vs. Predicted
plt.subplot(1, 2, 1)
plt.scatter(actual_sec, pred_sec, alpha=0.2, color='#1f77b4', s=10)

# red line of perfection (x=y)
max_val = 10000 # Zoom on the first 2.7 hours 
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Prediction')

plt.title('Reaklity vs Prevision of the model', fontsize=14)
plt.xlabel('Actual Duration (Seconds)', fontsize=12)
plt.ylabel('Prevision of duration (Seconds)', fontsize=12)
plt.xlim(0, max_val)
plt.ylim(0, max_val)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# GRAPH 2: THE RESIDUALS
plt.subplot(1, 2, 2)
plt.scatter(actual_sec, errors, alpha=0.2, color='#ff7f0e', s=10)

# Zero line 
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Error')

plt.title('Distribution of errors (residuals)', fontsize=14)
plt.xlabel('Real Duration(Seconds)', fontsize=12)
plt.ylabel('Errors (Seconds)', fontsize=12)
plt.xlim(0, max_val)
# Limit the Y-axis to prevent scaling issues from a 10-hour outlier.
plt.ylim(-4000, 4000) 
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Create a wide figure with 3 side-by-side plots.
plt.figure(figsize=(18, 5))

# Plot 1: Error Distribution
plt.subplot(1, 3, 1)
# Plottiamo solo il 99% dei dati normali, ignorando i mega-outlier visivamente
limite_errore = analysis['absolute_error'].quantile(0.95) 
sns.histplot(analysis[analysis['absolute_error'] < limite_errore]['absolute_error'], bins=50, kde=True, color='red')
plt.title('1. Seconds of error')
plt.xlabel('Absolute error (Second)')
plt.ylabel('Number of rides')

# Plot 2: Actual vs. Predicted (The Ground Truth)
plt.subplot(1, 3, 2)
limite_reale = analysis['last_in_sec'].quantile(0.99)
dati_puliti = analysis[(analysis['last_in_sec'] < limite_reale) & (analysis['forecast_sec'] < limite_reale)]

plt.scatter(dati_puliti['last_in_sec'], dati_puliti['forecast_sec'], alpha=0.1, color='blue')
#Red dashed line: Perfect Prediction Line
plt.plot([0, limite_reale], [0, limite_reale], 'r--', linewidth=2) 
plt.title('2. Prevision vs Reality')
plt.xlabel('Actual duration (Seconds)')
plt.ylabel('Precision of the models (Seconds)')

# Plot 3: Where does it fail most? (Error vs. Distance)
plt.subplot(1, 3, 3)
limite_distanza = analysis['distance_km'].quantile(0.99)
dati_dist = analysis[analysis['distance_km'] < limite_distanza]

plt.scatter(dati_dist['distance_km'], dati_dist['absolute_error'], alpha=0.1, color='purple')
plt.title('3. Does the error worsen on long hauls??')
plt.xlabel('Distance (km)')
plt.ylabel('Absolute error (Seconds)')

plt.tight_layout()
plt.show()

## let's now predict the test set 



In [ ]:
print('-'*80)
print('TESTSET')
print('-'*80)
# EDA
# test set 
#statical description about the test set 
print(test_set.describe())
print('-'*80)
# infos about columns and kind of datas 
print(test_set.info())
print('-'*80)
# printing first rows
print(test_set.head())

In [ ]:
test_ids = test_set['id'].copy()

cols_to_drop = [target, 'speed_km_h', 'is_test', 'dropoff_datetime', 'id']

X_train = train.drop(columns=cols_to_drop, errors='ignore')
y_train = np.log1p(y_train_clipped)

X_val = validation.drop(columns=cols_to_drop, errors='ignore')
y_val = np.log1p(y_val_clipped)

X_internal_test = internal_test.drop(columns=cols_to_drop, errors='ignore')

X_final = test_set[X_train.columns]
forecast_log = xgb_model.predict(X_final)
forecast_seconds = np.expm1(forecast_log)

submission = pd.DataFrame({
    'id': test_ids,
    'trip_duration': forecast_seconds
})
submission.to_csv('submission.csv', index=False)
print(f"submission saved {len(submission)} rows")
print(submission.head())

